# 图数据库智能体搭建

# 1. 初始化环境变量

In [1]:
%%writefile test.py
import dotenv, os
os.environ["NEO4J_URI"] = dotenv.get_key(dotenv.find_dotenv(), "NEO4J_URI")
os.environ["NEO4J_USERNAME"] = dotenv.get_key(dotenv.find_dotenv(), "NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = dotenv.get_key(dotenv.find_dotenv(), "NEO4J_PASSWORD")


Overwriting test.py


# 2. 连接Neo4j数据库

In [2]:
%%writefile -a test.py

from langchain_neo4j import Neo4jGraph
graph = Neo4jGraph(
    url=os.environ["NEO4J_URI"],
    username=os.environ["NEO4J_USERNAME"],
    password=os.environ["NEO4J_PASSWORD"],
)

Appending to test.py


# 3. 加载大模型

In [3]:
%%writefile -a test.py

from agent_lib import qianwen_llm
llm = qianwen_llm

Appending to test.py


# 4. 智能体工具设计

#### **工具1**  智能体初始化
该工具主要用于生成根节点

In [4]:
%%writefile -a test.py

# 智能体初始化
from langchain.tools import tool
@tool
def tool_graph_initializer():
    """Initialize graph database. 
    """
    root_nodes = [
        {'label': '基础设施', 'name': '路基'},
        {'label': '基础设施', 'name': '桥梁'},
        {'label': '基础设施', 'name': '隧道'},
        {'label': '基础设施', 'name': '轨道'},
    ]
    for node in root_nodes:
        if not graph.query('MATCH (n:基础设施 {name:$name}) RETURN n', {'name': node['name']}):
            graph.query('create (n:基础设施 {name:$name}) RETURN n', {'name': node['name']})
    return graph.query("MATCH (n) RETURN n")

Appending to test.py


#### **工具2** 总图更新
对于节点关系的抽取，主要分为以下几个步骤
1. 子图的抽取，包含节点、关系以及各自的属性（subgraph_extractor)
2. 子图子图自检，确保合法（graph_validator)，此步骤需要设计用户确定步骤。
3. 写入Neo4j（graph_writter)

In [5]:
%%writefile -a test.py
from pydantic import BaseModel, Field
from typing_extensions import GraphState

class KGNode(BaseModel):
    id: str = Field(..., description="子图内唯一ID，例如 n1, n2")
    name: str
    type: str
    aliases: List[str] = Field(default_factory=list)
    attrs: Dict[str, Any] = Field(default_factory=dict)

class KGEdge(BaseModel):
    source: str
    target: str
    type: str
    confidence: float = Field(..., ge=0.0, le=1.0)
    evidence: str = Field(..., description="支撑该关系的原句/片段")
    attrs: Dict[str, Any] = Field(default_factory=dict)

class Subgraph(BaseModel):
    nodes: List[KGNode]
    edges: List[KGEdge]
    meta: Dict[str, Any] = Field(default_factory=dict)

class State(TypedDict):
    subgraph: Subgraph
    validity: bool


# 大模型抽取子图
def subgraph_extractor(State):
    # TODO
    pass

def graph_validator(State):
    # TODO
    pass

def graph_writter(State):
    # TODO
    pass

from langgraph.graph import StateGraph, START, END
builder = StateGraph(State)

builder.add_node("subgraph_extractor", subgraph_extractor)
builder.add_node("graph_validator",graph_validator)
builder.add_node("graph_writter", graph_writter)

builder.add_edge(START, "subgraph_extractor")
builder.add_edge("subgraph_extractor", "graph_validator")
builder.add_edge("graph_validator", "graph_writter")
builder.add_edge("graph_writter", END)
chain = builder.compile()
from IPython.display import Image, display
Image(chain.get_graph().draw_mermaid_png())



ImportError: cannot import name 'GraphState' from 'typing_extensions' (G:\学习\研究生\python_code\RailwayMaintenanceAgent\.venv\Lib\site-packages\typing_extensions.py)